# Portfolio risk and drawdown analysis

Use Tyche close-price batches to build a compact risk report: performance, volatility, drawdowns, correlations, and equal-weight risk contributions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tyche_client import TycheClient

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Pull an investable cross-asset basket

`price_matrix()` aligns closes by date and now chunks lists larger than Tyche's 50-symbol request limit.

In [ ]:
tickers = ["SPY", "QQQ", "IWM", "TLT", "GLD", "DBC"]
prices = client.price_matrix(tickers, start="2021-01-01").ffill().dropna()
returns = prices.pct_change().dropna()

(prices / prices.iloc[0]).plot(figsize=(11, 5), title="Growth of $1")
plt.ylabel("normalized value")
plt.show()

## Annualized return, risk, and maximum drawdown

These are historical estimates, not forecasts. The return estimate is geometric; volatility uses 252 trading days.

In [ ]:
years = (prices.index[-1] - prices.index[0]).days / 365.25
annual_return = (prices.iloc[-1] / prices.iloc[0]) ** (1 / years) - 1
annual_volatility = returns.std() * np.sqrt(252)
drawdown = prices / prices.cummax() - 1

summary = pd.DataFrame({
    "annual_return": annual_return,
    "annual_volatility": annual_volatility,
    "return_to_risk": annual_return / annual_volatility,
    "max_drawdown": drawdown.min(),
}).sort_values("return_to_risk", ascending=False)
summary.style.format("{:.1%}", subset=["annual_return", "annual_volatility", "max_drawdown"]).format("{:.2f}", subset=["return_to_risk"])

In [ ]:
drawdown.plot(figsize=(11, 5), title="Drawdowns from each asset's prior high")
plt.ylabel("drawdown")
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

## Correlation and portfolio risk contribution

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
fig.colorbar(im, ax=ax, label="daily-return correlation")
ax.set_title("Diversification map")
plt.tight_layout()

In [ ]:
weights = pd.Series(1 / returns.shape[1], index=returns.columns)
cov = returns.cov() * 252
portfolio_vol = np.sqrt(weights @ cov @ weights)
risk_contribution = weights * (cov @ weights) / portfolio_vol

pd.DataFrame({
    "weight": weights,
    "volatility_contribution": risk_contribution,
    "share_of_total_risk": risk_contribution / portfolio_vol,
}).sort_values("share_of_total_risk", ascending=False).style.format("{:.1%}")